# End-to-end pipeline benchmark on 91 real parking-camera clips

Original notebook that produced `pipelinetestresults_optimized.csv` and the 65.9% / 46.6 FPS figures. The same logic is refactored into `src/anpr/`.


In [1]:
import os
import re
import time
import yaml
import cv2
import torch
import numpy as np
import pandas as pd
import onnxruntime as ort
from ultralytics import YOLO
from collections import Counter, defaultdict

In [2]:
torch.backends.cudnn.benchmark = True
cv2.setUseOptimized(True)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Torch CUDA:", torch.version.cuda)

print("ORT version:", ort.__version__)
print("ORT available providers:", ort.get_available_providers())

Torch: 2.12.0.dev20260305+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070
Torch CUDA: 12.8
ORT version: 1.24.3
ORT available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [3]:
CLIPS_DIR = r"clips"
CAR_MODEL_PATH = r"yolo11n.pt"
PLATE_MODEL_PATH = r"yolo11n_plate_best.pt"
OCR_MODEL_PATH = r"ru_plate_cct_xs.onnx"
OCR_CONFIG_PATH = r"ru_plate_config.yaml"

COCO_VEHICLE_CLASSES = [2, 3, 5, 7]

SAMPLE_EVERY_N = 2
OCR_EVERY_N = 1
CONF_CAR = 0.45
CONF_PLATE = 0.60
CAR_IMGSZ = 960
PLATE_IMGSZ = 320
MAX_CARS = 2
MIN_CAR_AREA_RATIO = 0.02
MIN_PLATE_AREA = 420
MIN_PLATE_AREA_RATIO_IN_CAR = 0.004
MAX_PLATE_AREA_RATIO_IN_CAR = 0.45
MIN_PLATE_ASPECT = 1.6
MAX_PLATE_ASPECT = 8.5
MIN_OCR_CONF = 0.18
AREA_KEEP_RATIO = 0.35
EARLY_STOP_VOTES = 3
EARLY_STOP_SCORE = 1.9
USE_CLAHE = True

TEST_CLIPS = []

In [4]:
USE_CUDA = torch.cuda.is_available()
DEVICE = 0 if USE_CUDA else "cpu"
USE_HALF = bool(USE_CUDA)

car_detector = YOLO(CAR_MODEL_PATH)
plate_detector = YOLO(PLATE_MODEL_PATH)

if USE_CUDA:
    car_detector.to("cuda")
    plate_detector.to("cuda")

with open(OCR_CONFIG_PATH, "r", encoding="utf-8") as f:
    ocr_config = yaml.safe_load(f)

MY_ALPHABET = ocr_config["alphabet"]
MY_PAD_CHAR = ocr_config.get("pad_char", "_")

sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
sess_options.intra_op_num_threads = max(1, (os.cpu_count() or 4) // 2)
sess_options.inter_op_num_threads = 1

providers = ["CPUExecutionProvider"]
if "CUDAExecutionProvider" in ort.get_available_providers():
    if hasattr(ort, "preload_dlls"):
        ort.preload_dlls()
    providers = [
        ("CUDAExecutionProvider", {
            "device_id": 0,
            "cudnn_conv_use_max_workspace": "1",
            "use_tf32": "1",
            "do_copy_in_default_stream": "1"
        }),
        "CPUExecutionProvider"
    ]

ocr_session = ort.InferenceSession(
    OCR_MODEL_PATH,
    sess_options=sess_options,
    providers=providers
)

input_name = ocr_session.get_inputs()[0].name
expected_type = ocr_session.get_inputs()[0].type

print("YOLO device:", "cuda" if USE_CUDA else "cpu")
print("ORT session providers:", ocr_session.get_providers())
print("ORT input type:", expected_type)

Skip loading CUDA and cuDNN DLLs since torch is imported.
YOLO device: cuda
ORT session providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']
ORT input type: tensor(uint8)


In [5]:
PLATE_LETTERS = set("ABEKMHOPCTYX")
DIGITS = set("0123456789")
PLATE_REGEX = re.compile(r"^[A-Z]\d{3}[A-Z]{2}\d{2,3}$")
LETTER_POS = {0, 4, 5}

LETTER_TO_DIGIT = {
    "O":"0","Q":"0","D":"0","I":"1","L":"1","J":"1",
    "Z":"7","T":"7","V":"7","Y":"7","S":"5","B":"8",
    "A":"4","G":"6"
}
TO_ALLOWED_LETTER = {
    "0":"O","D":"O","Q":"O","4":"A","7":"T","I":"T","1":"T",
    "8":"B","5":"S","S":"C","U":"Y","V":"Y","N":"H","R":"P",
    "3":"E","F":"E","J":"T","W":"M"
}
FORCE_DIGIT_MAP = {
    "V":"7","Y":"7","Z":"7","T":"7","S":"5","G":"6",
    "B":"8","I":"1","L":"1","J":"1","O":"0","Q":"0","D":"0","A":"4"
}

def normalize_plate_text(text: str) -> str:
    text = text.upper()
    replacements = {
        "А":"A","В":"B","Е":"E","К":"K","М":"M","Н":"H",
        "О":"O","Р":"P","С":"C","Т":"T","У":"Y","Х":"X"
    }
    for cyr, lat in replacements.items():
        text = text.replace(cyr, lat)
    return re.sub(r"[^A-Z0-9]", "", text)

def to_digit(ch):
    return ch if ch in DIGITS else LETTER_TO_DIGIT.get(ch, ch)

def to_letter(ch):
    return ch if ch in PLATE_LETTERS else TO_ALLOWED_LETTER.get(ch, ch)

def fix_prefix6(raw6):
    chars = list(raw6)
    changes = 0
    for i, ch in enumerate(chars):
        new_ch = to_letter(ch) if i in LETTER_POS else to_digit(ch)
        if new_ch != ch:
            changes += 1
        chars[i] = new_ch
    ok = (
        chars[0] in PLATE_LETTERS and
        chars[1] in DIGITS and
        chars[2] in DIGITS and
        chars[3] in DIGITS and
        chars[4] in PLATE_LETTERS and
        chars[5] in PLATE_LETTERS
    )
    return "".join(chars), ok, changes

def force_digits(s):
    result = list(s)
    for i in [1, 2, 3, 6, 7, 8]:
        if i < len(result) and result[i] in FORCE_DIGIT_MAP:
            result[i] = FORCE_DIGIT_MAP[result[i]]
    return "".join(result)

def postprocess_plate(text: str) -> str:
    if not text:
        return ""
    t = re.sub(r"[^0-9A-Z]", "", text.upper())
    for noise in ["RUSSIA", "RUSE", "RUS", "PUS", "PVS", "RVS", "RU5", "PU5", "R0S", "KUS", "PLS"]:
        t = t.replace(noise, "")

    if len(t) < 6:
        return t
    if len(t) == 6 and t[5] in "0123456789":
        t = t[:5] + "T" + t[5:]
    if len(t) < 8:
        return t

    best = None
    for s in range(0, len(t) - 5):
        raw6 = t[s:s+6]
        fixed6, ok, chg6 = fix_prefix6(raw6)
        if not ok:
            continue

        tail = t[s+6:]
        tail_digits = []
        tail_changes = 0
        for ch in tail:
            d = to_digit(ch)
            if d in DIGITS:
                tail_digits.append(d)
                if d != ch:
                    tail_changes += 1

        for region_len in [3, 2]:
            if len(tail_digits) >= region_len:
                cand = fixed6 + "".join(tail_digits[:region_len])
                if PLATE_REGEX.match(cand):
                    changes = chg6 + tail_changes
                    if best is None or changes < best[0]:
                        best = (changes, cand)

    return force_digits(best[1]) if best else t

def ctc_decode(preds, alphabet, pad_char):
    pred_indices = np.argmax(preds[0], axis=-1)
    pad_idx = alphabet.index(pad_char) if pad_char in alphabet else len(alphabet) - 1
    return "".join(alphabet[idx] for idx in pred_indices if idx != pad_idx)

In [6]:
def softmax_np(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x, axis=-1, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=-1, keepdims=True)


def ctc_decode_with_conf(preds, alphabet, pad_char):
    logits = preds[0]
    probs = softmax_np(logits)
    pred_indices = np.argmax(probs, axis=-1)
    pad_idx = alphabet.index(pad_char) if pad_char in alphabet else len(alphabet) - 1

    chars = []
    confs = []
    for t, idx in enumerate(pred_indices):
        if idx != pad_idx:
            chars.append(alphabet[idx])
            confs.append(float(probs[t, idx]))

    text = "".join(chars)
    score = float(np.mean(confs)) if confs else 0.0
    return text, score


def enhance_plate_crop(plate_crop):
    plate_gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
    if USE_CLAHE:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
        plate_gray = clahe.apply(plate_gray)
    small = cv2.resize(plate_gray, (140, 70), interpolation=cv2.INTER_LINEAR)
    return small


def ocr_plate(plate_crop):
    try:
        small = enhance_plate_crop(plate_crop)

        if "float" in expected_type:
            img_input = small.astype(np.float32)[None, :, :, None] / 255.0
        else:
            img_input = small[None, :, :, None]

        preds = ocr_session.run(None, {input_name: img_input})[0]
        raw, conf = ctc_decode_with_conf(preds, MY_ALPHABET, MY_PAD_CHAR)
        pred = postprocess_plate(normalize_plate_text(raw))
        return pred, conf
    except Exception:
        return "", 0.0

In [7]:
def is_valid_plate_shape(w, h, car_area=None):
    if w <= 0 or h <= 0:
        return False
    area = w * h
    aspect = w / max(h, 1)
    if area < MIN_PLATE_AREA:
        return False
    if not (MIN_PLATE_ASPECT <= aspect <= MAX_PLATE_ASPECT):
        return False
    if car_area is not None and car_area > 0:
        ratio = area / car_area
        if ratio < MIN_PLATE_AREA_RATIO_IN_CAR or ratio > MAX_PLATE_AREA_RATIO_IN_CAR:
            return False
    return True


def is_soft_plate_text(text):
    if not text:
        return False
    text = postprocess_plate(normalize_plate_text(text))
    return len(text) >= 6


def is_strict_plate_text(text):
    if not text:
        return False
    text = postprocess_plate(normalize_plate_text(text))
    return bool(PLATE_REGEX.match(text))


def aggregate_clip_prediction(all_detections):
    if not all_detections:
        return "NOTDETECTED"

    prepared = []
    for d in all_detections:
        text = postprocess_plate(normalize_plate_text(d.get("text", "")))
        if not is_soft_plate_text(text):
            continue
        dd = dict(d)
        dd["text"] = text
        dd["ocr_conf"] = float(d.get("ocr_conf", 0.0))
        dd["area"] = float(d.get("area", 0.0))
        prepared.append(dd)

    if not prepared:
        return "NOTDETECTED"

    strict = [d for d in prepared if is_strict_plate_text(d["text"]) and d["ocr_conf"] >= MIN_OCR_CONF]
    soft = [d for d in prepared if d["ocr_conf"] >= max(0.08, MIN_OCR_CONF * 0.5)]

    working = strict if strict else soft if soft else prepared
    max_area = max(d["area"] for d in working) if working else 0.0
    large = [d for d in working if max_area <= 0 or d["area"] >= max_area * AREA_KEEP_RATIO]
    if not large:
        large = working

    text_weights = defaultdict(float)
    text_counts = Counter()
    for d in large:
        area_norm = d["area"] / max(max_area, 1.0)
        weight = 0.50 * area_norm + 0.50 * max(d["ocr_conf"], 0.05)
        text_weights[d["text"]] += weight
        text_counts[d["text"]] += 1

    regex_candidates = [t for t in text_weights if is_strict_plate_text(t)]
    if regex_candidates:
        best_full = max(regex_candidates, key=lambda t: (text_counts[t], text_weights[t], len(t)))
        if text_counts[best_full] >= 2:
            return best_full

    max_len = max(len(d["text"]) for d in large)
    final_chars = []
    for pos in range(max_len):
        votes = defaultdict(float)
        for d in large:
            text = d["text"]
            if pos >= len(text):
                continue
            area_norm = d["area"] / max(max_area, 1.0)
            char_w = 0.55 * max(d["ocr_conf"], 0.05) + 0.45 * area_norm
            votes[text[pos]] += char_w
        if votes:
            final_chars.append(max(votes.items(), key=lambda kv: kv[1])[0])

    best = postprocess_plate("".join(final_chars))
    if is_strict_plate_text(best):
        return best

    fallback = max(text_weights.keys(), key=lambda t: (is_strict_plate_text(t), text_counts[t], text_weights[t], len(t)))
    return fallback if fallback else "NOTDETECTED"

In [8]:
def process_clip_fast(
    clip_path,
    sample_every_n=SAMPLE_EVERY_N,
    ocr_every_n=OCR_EVERY_N,
    conf_car=CONF_CAR,
    conf_plate=CONF_PLATE,
    car_imgsz=CAR_IMGSZ,
    plate_imgsz=PLATE_IMGSZ,
    max_cars=MAX_CARS,
    min_car_area_ratio=MIN_CAR_AREA_RATIO,
    min_plate_area=MIN_PLATE_AREA,
    early_stop_votes=EARLY_STOP_VOTES,
):
    cap = cv2.VideoCapture(clip_path)
    if not cap.isOpened():
        return "NOTDETECTED", 0, 0, 0.0

    all_detections = []
    text_votes = Counter()
    text_scores = defaultdict(float)
    frame_idx = 0
    processed_frames = 0
    total_time = 0.0

    while True:
        ok = cap.grab()
        if not ok:
            break

        if frame_idx % sample_every_n != 0:
            frame_idx += 1
            continue

        ok, frame = cap.retrieve()
        if not ok:
            frame_idx += 1
            continue

        t0 = time.perf_counter()
        h, w = frame.shape[:2]
        frame_area = h * w

        car_results = car_detector.predict(
            source=frame,
            conf=conf_car,
            imgsz=car_imgsz,
            classes=COCO_VEHICLE_CLASSES,
            max_det=max_cars,
            half=USE_HALF,
            device=DEVICE,
            verbose=False
        )[0]

        cars = []
        for box in car_results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            x1 = max(0, x1); y1 = max(0, y1)
            x2 = min(w, x2); y2 = min(h, y2)
            if x2 <= x1 or y2 <= y1:
                continue

            area = (x2 - x1) * (y2 - y1)
            if area < frame_area * min_car_area_ratio:
                continue

            conf_box = float(box.conf[0]) if hasattr(box, 'conf') else 0.0
            cars.append((area, conf_box, x1, y1, x2, y2))

        cars.sort(reverse=True)
        do_ocr_this_frame = ((processed_frames % ocr_every_n) == 0)

        for _, _, x1, y1, x2, y2 in cars[:max_cars]:
            car_crop = frame[y1:y2, x1:x2]
            if car_crop.size == 0:
                continue

            car_area = car_crop.shape[0] * car_crop.shape[1]
            plate_results = plate_detector.predict(
                source=car_crop,
                conf=conf_plate,
                imgsz=plate_imgsz,
                max_det=2,
                half=USE_HALF,
                device=DEVICE,
                verbose=False
            )[0]

            if len(plate_results.boxes) == 0:
                continue

            candidates = []
            for pbox in plate_results.boxes:
                px1, py1, px2, py2 = map(int, pbox.xyxy[0])
                px1 = max(0, px1); py1 = max(0, py1)
                px2 = min(car_crop.shape[1], px2); py2 = min(car_crop.shape[0], py2)
                if px2 <= px1 or py2 <= py1:
                    continue

                pw, ph = (px2 - px1), (py2 - py1)
                area = pw * ph
                if area < min_plate_area:
                    continue
                if not is_valid_plate_shape(pw, ph, car_area):
                    continue

                det_conf = float(pbox.conf[0]) if hasattr(pbox, 'conf') else 0.0
                aspect = pw / max(ph, 1)
                score = 0.65 * det_conf + 0.25 * min(aspect / 4.0, 1.5) + 0.10 * min(area / 2500.0, 1.5)
                candidates.append((score, area, px1, py1, px2, py2, det_conf))

            if not candidates:
                continue

            candidates.sort(reverse=True)
            _, best_area, px1, py1, px2, py2, plate_det_conf = candidates[0]
            gx1, gy1 = x1 + px1, y1 + py1
            gx2, gy2 = x1 + px2, y1 + py2
            plate_crop = car_crop[py1:py2, px1:px2]

            pred = ""
            ocr_conf = 0.0
            if do_ocr_this_frame and plate_crop.size != 0:
                pred, ocr_conf = ocr_plate(plate_crop)

            if pred:
                pred = postprocess_plate(normalize_plate_text(pred))
                if not is_soft_plate_text(pred):
                    pred = ""

            if pred:
                center = ((gx1 + gx2) // 2, (gy1 + gy2) // 2)
                all_detections.append({
                    "text": pred,
                    "area": best_area,
                    "center": center,
                    "ocr_conf": ocr_conf,
                    "plate_det_conf": plate_det_conf
                })
                text_votes[pred] += 1
                text_scores[pred] += 0.5 * max(ocr_conf, 0.05) + 0.5 * min(best_area / max(car_area, 1), 0.2)

                if is_strict_plate_text(pred) and text_votes[pred] >= early_stop_votes and text_scores[pred] >= EARLY_STOP_SCORE:
                    total_time += time.perf_counter() - t0
                    processed_frames += 1
                    cap.release()
                    return pred, frame_idx + 1, processed_frames, total_time

        total_time += time.perf_counter() - t0
        frame_idx += 1
        processed_frames += 1

    cap.release()
    final_plate = aggregate_clip_prediction(all_detections)
    return final_plate, frame_idx, processed_frames, total_time

In [9]:
all_clips = sorted([f for f in os.listdir(CLIPS_DIR) if f.lower().endswith(".avi")])
clips_to_run = TEST_CLIPS if TEST_CLIPS else all_clips

results = []
char_acc_sum = 0.0

for i, clip_name in enumerate(clips_to_run, start=1):
    clip_path = os.path.join(CLIPS_DIR, clip_name)
    recognized, frames_seen, processed_frames, elapsed = process_clip_fast(clip_path)

    gt = os.path.splitext(clip_name)[0]
    gt = gt.split("_", 1)[-1] if "_" in gt else gt[2:] if len(gt) > 2 and gt[:2].isdigit() else gt
    gt = re.sub(r"[^A-Z0-9А-Яа-я]", "", gt.upper())
    gt = normalize_plate_text(gt)

    rec_clean = re.sub(r"[^A-Z0-9]", "", recognized.upper()) if recognized else ""
    matches = sum(p == g for p, g in zip(rec_clean, gt)) if gt and rec_clean and rec_clean != "NOTDETECTED" else 0
    char_acc = matches / max(len(gt), len(rec_clean), 1) * 100.0
    char_acc_sum += char_acc

    results.append({
        "clip": clip_name,
        "gt": gt,
        "recognized": recognized,
        "char_acc": round(char_acc, 1),
        "frames_seen": frames_seen,
        "processed_frames": processed_frames,
        "elapsed_sec": round(elapsed, 3),
    })

    print(f"[{i:02d}/{len(clips_to_run)}] {clip_name} -> {recognized} | proc={processed_frames} | time={elapsed:.2f}s")

results_df = pd.DataFrame(results)
results_df.to_csv("pipelinetestresults_optimized.csv", index=False, encoding="utf-8-sig")
results_df.head()

[01/91] 01_T3•••KK••.avi -> T3•••KK•• | proc=95 | time=10.31s
[02/91] 02_P1•••PM••.avi -> T2•••HH•• | proc=72 | time=8.74s
[03/91] 03_B5•••OB••.avi -> B5•••OB•• | proc=246 | time=8.15s
[04/91] 04_M8•••KM••.avi -> K1•••YY•• | proc=764 | time=18.17s
[05/91] 05_T6•••EK••.avi -> NOTDETECTED | proc=230 | time=6.22s
[06/91] 06_H7•••AA••.avi -> H7•••AA•• | proc=143 | time=3.09s
[07/91] 07_M8•••KM••.avi -> NOTDETECTED | proc=67 | time=2.57s
[08/91] 08_X7•••BE••.avi -> X7•••BE•• | proc=104 | time=2.38s
[09/91] 09_C4•••TA••.avi -> C4•••TA•• | proc=122 | time=2.37s
[10/91] 10_A8•••EE••.avi -> A8•••EE•• | proc=86 | time=1.95s
[11/91] 11_H8•••OM••.avi -> H8•••OM•• | proc=50 | time=1.38s
[12/91] 12_K5•••BT••.avi -> K5•••BT•• | proc=71 | time=1.61s
[13/91] 13_A8•••HE••.avi -> NOTDETECTED | proc=72 | time=1.38s
[14/91] 14_A1•••OE••.avi -> A1•••OE•• | proc=96 | time=2.17s
[15/91] 15_O1•••TA••.avi -> O1•••TA•• | proc=82 | time=1.68s
[16/91] 16_H4•••PB••.avi -> H4•••PB•• | proc=411 | time=10.28s
[17/91] 

,clip,gt,recognized,char_acc,frames_seen,processed_frames,elapsed_sec
0,01_T3•••KK••.avi,T3•••KK••,T3•••KK••,100.0,189,95,10.308
1,02_P1•••PM••.avi,P1•••PM••,T2•••HH••,11.1,144,72,8.739
2,03_B5•••OB••.avi,B5•••OB••,B5•••OB••,100.0,492,246,8.146
3,04_M8•••KM••.avi,M8•••KM••,K1•••YY••,0.0,1528,764,18.169
4,05_T6•••EK••.avi,T6•••EK••,NOTDETECTED,0.0,460,230,6.218


In [11]:
results_df = pd.read_csv("pipelinetestresults_optimized.csv")

total_clips = len(results_df)
correct = int((results_df["gt"] == results_df["recognized"]).sum())
notdetected = int((results_df["recognized"] == "NOTDETECTED").sum())
wrong = total_clips - correct - notdetected
char_acc_avg = float(results_df["char_acc"].mean()) if total_clips else 0.0

total_processed_frames = int(results_df["processed_frames"].sum())
total_time = float(results_df["elapsed_sec"].sum())
avg_fps = total_processed_frames / total_time if total_time > 0 else 0.0

print("-" * 60)
print("ALPR optimized")
print("-" * 60)
print("clips:", total_clips)
print("correct:", correct, f"({correct / total_clips * 100:.1f}%)" if total_clips else "(0.0%)")
print("wrong:", wrong)
print("notdetected:", notdetected)
print("avg char acc:", f"{char_acc_avg:.1f}%")
print("processed frames:", total_processed_frames)
print("total time:", f"{total_time:.2f}s")
print("avg processed FPS:", f"{avg_fps:.2f}")
print("-" * 60)

------------------------------------------------------------
ALPR optimized
------------------------------------------------------------
clips: 91
correct: 60 (65.9%)
wrong: 22
notdetected: 9
avg char acc: 80.7%
processed frames: 16714
total time: 358.35s
avg processed FPS: 46.64
------------------------------------------------------------
